# 00 Project Overview

Dieses Notebook definiert den fachlichen Projekt-Scope fuer die Football-Weather-Big-Data-Analyse. Es legt fest, welche Turniere, Match-Anzahlen und Zielmetriken in der Pipeline verwendet werden.

## Forschungsfrage

Wie haengen Wetterbedingungen und Teamstaerke mit Spielstil und Chancenqualitaet bei internationalen Fussballturnieren zusammen?

Die Hauptanalyse-Ebene ist `team_match`: Ein Spiel wird als zwei Team-Beobachtungen modelliert, eine Zeile pro Team.

## Methodischer Ansatz

Der Scope wird als reproduzierbarer Datenvertrag behandelt: Turniere, Wettbewerbe und Saisons werden strukturiert erfasst und als maschinenlesbare JSON-Datei gespeichert. Dadurch verwenden alle Pipeline-Schritte dieselbe fachliche Grundlage.

Die Analyse nutzt tabellarische Datenprodukte und eine klare Trennung zwischen Scope-Definition, Datenaufnahme, Feature Engineering und Auswertung.

In [1]:
import json

from project_paths import project_root as get_project_root

base_path = get_project_root()
scope_path = base_path / 'data' / 'reference' / 'tournament_scope.json'
with scope_path.open('r', encoding='utf-8') as f:
    scope = json.load(f)

tournaments = scope['tournaments']
metrics = scope['target_metrics']

print(f"Scope loaded from: {scope_path}")


Scope loaded from: /home/jovyan/work/data/reference/tournament_scope.json


## Turnier-Scope

In [2]:
[
    {
        'scope_id': tournament['scope_id'],
        'short_name': tournament['short_name'],
        'competition_name': tournament['competition_name'],
        'season_name': tournament['season_name'],
        'expected_matches': tournament['expected_matches'],
        'expected_team_match_rows': tournament['expected_team_match_rows'],
    }
    for tournament in tournaments
]

[{'scope_id': 'world_cup_2018',
  'short_name': 'WM 2018',
  'competition_name': 'FIFA World Cup',
  'season_name': '2018',
  'expected_matches': 64,
  'expected_team_match_rows': 128},
 {'scope_id': 'world_cup_2022',
  'short_name': 'WM 2022',
  'competition_name': 'FIFA World Cup',
  'season_name': '2022',
  'expected_matches': 64,
  'expected_team_match_rows': 128},
 {'scope_id': 'euro_2020',
  'short_name': 'Euro 2020',
  'competition_name': 'UEFA Euro',
  'season_name': '2020',
  'expected_matches': 51,
  'expected_team_match_rows': 102},
 {'scope_id': 'euro_2024',
  'short_name': 'Euro 2024',
  'competition_name': 'UEFA Euro',
  'season_name': '2024',
  'expected_matches': 51,
  'expected_team_match_rows': 102},
 {'scope_id': 'afcon_2023',
  'short_name': 'AFCON 2023',
  'competition_name': 'Africa Cup of Nations',
  'season_name': '2023',
  'expected_matches': 52,
  'expected_team_match_rows': 104},
 {'scope_id': 'copa_america_2024',
  'short_name': 'Copa America 2024',
  'compe

## Erwartete Gesamtgroesse

In [3]:
expected_matches = sum(tournament['expected_matches'] for tournament in tournaments)
expected_team_rows = sum(tournament['expected_team_match_rows'] for tournament in tournaments)

assert expected_matches == scope['expected_totals']['matches']
assert expected_team_rows == scope['expected_totals']['team_match_rows']
assert all(
    tournament['expected_team_match_rows'] == tournament['expected_matches'] * 2
    for tournament in tournaments
)

{
    'tournaments': len(tournaments),
    'expected_matches': expected_matches,
    'expected_team_match_rows': expected_team_rows,
    'project_level': scope['project_level'],
}

{'tournaments': 6,
 'expected_matches': 314,
 'expected_team_match_rows': 628,
 'project_level': 'team_match'}

## Team-Match-Vertrag

- Ein Match erzeugt genau zwei Team-Match-Zeilen.
- Jede Zeile beschreibt ein Team gegen einen Gegner in einem konkreten Spiel.
- Wetter- und Elo-Features werden an diese Team-Match-Zeilen gejoint.
- Data-Quality-Check fuer Feature-Notebooks: Jede `match_id` muss genau zwei Team-Zeilen besitzen.

## Zielmetriken

In [4]:
metrics

[{'name': 'team_xg',
  'description': 'Sum of shot xG for the team in one match.'},
 {'name': 'shots', 'description': 'Number of team shots in one match.'},
 {'name': 'xg_per_shot',
  'description': 'team_xg divided by shots; null if shots is zero.'},
 {'name': 'passes', 'description': 'Number of team pass events in one match.'},
 {'name': 'pass_completion_rate',
  'description': 'Completed passes divided by all passes; null if passes is zero.'},
 {'name': 'pressures',
  'description': 'Number of team pressure events in one match.'},
 {'name': 'counterpressures',
  'description': 'Number of team counterpressure events in one match.'},
 {'name': 'duels', 'description': 'Number of team duel events in one match.'},
 {'name': 'long_ball_share',
  'description': 'Long passes divided by all passes; null if passes is zero.'}]

## Notebook-Vertrag

Alle Pipeline-Notebooks lesen `data/reference/tournament_scope.json` als zentrale Scope-Quelle. Die ausfuehrliche Dokumentation steht in `docs/project_scope.md`, die Pipeline-Aufteilung in `docs/notebook_data_flow.md`.